In [2]:
import pandas as pd
from codes.utils import Config

In [3]:
info = pd.read_csv(Config.DIR_HOME / 'docs' / 'chemical_effect_human.txt', delimiter='\t')
info[info.isna()] = ''
info

/var/folders/g3/sdb5v4350551p2bbc02jj5xdwg5ncq/T/ipykernel_76213/283433957.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  info[info.isna()] = ''


,Other,Toxicity,"Aspirin, 50-78-2","Paracetamol, 103-90-2","Metformin, 657-24-9","Atorvastatin, 134523-00-5","Prednisone, 53-03-2","Dioxin, 1746-01-6","Benzene, 71-43-2","Arsenite, 15502-74-6",...,"6PPQ (N-(1,3-dimethylbutyl)-N'-phenyl-p-phenylenediamine), 2754428-18-5","Phenanthrene, 85-01-8","TPHP (triphenylphosphate), 115-86-6","Lanthanum, 7439-91-0","Galaxolide, 1222-05-5","1,6-Dimethyl-3,4-dihydroisoquinoline, 91753-09-2","2-Methyl-4-(4-methylphenyl)-2,3-dihydro-1,5-benzothiazepine, 74148-62-2","N-Cyclopropylmethyl-10,11-dihydro-5H-dibenzo-(a,d)-cyclohepten-5-imine, 59864-46-9","4-Methyl-1,2-dihydrobenzo[f]isoquinoline, 29248-42-8","6-Methyl-2,5-diphenyl-6H-1,3,4-thiadiazine, 62625-70-1"
0,,Sub acute,Gastrointestinal bleeding,Hypothermia; Acidosis; Hepatocellular/liver in...,Acidosis,,Weight gain; Increased appetite,Altered liver function,,Fatty degeneration of the heart,...,,,,,,,,,,
1,,Chronic,Gastrointestinal bleeding; Anemia; Chronic kid...,,,,Osteoporosis; Weight gain; Increased appetite;...,Breast cancer; Endometrial cancer; Testis cancer,Anemia; Bone marrow suppression; Immune suppre...,Anemia; Leukopenia; Liver damage; Hyperpigment...,...,,,,,,,,,,
2,,Developmental,Gastroschisis,,,,,,,,...,,,,,,,,,,
3,,Immune/lymphatic system,Hypersensitivity/allergic reaction,,,,Edema; Immune suppression,Immune suppression,,,...,,,,,,,,,,
4,,Reproductive system,,,,,,Endometriosis; Decreased sperm count,,,...,,,,,,,,,,
5,,Nervous system/Neurological,Ototoxicity,Ototoxicity,Visual disturbances,Amblyopia; Dry eyes; Eye hemorrhage; Glaucoma,Delerium; Cataracts; Insomnia; Aggression,,Drowsiness; Dizziness; Headache; Vertigo; Trem...,,...,,,,,,,,,,
6,,Digestive system,Gastrointestinal bleeding; Ulcers; Autoimmune ...,Hepatocellular/Liver injury; Pancreatitis; Ora...,Nausea; Vomiting; Hepatocellular injury; Chole...,Constipation; Pancreatitis; Liver toxicity,Pancreatitis,,,,...,,,,,,,,,,
7,,Urinary system,,Acidosis,,,,,,,...,,,,,,,,,,
8,,Integumentary system,,Fixed drug eruption; Steven-Johnson syndrome,,Alopecia; Erythema multiforme,Alopecia,Chloracne,,,...,,,,,,,,,,
9,,Musculoskeletal system,,,,Rhabdomyloysis,Osteoporosis; Sarcopenia/Muscle weakness,,,,...,,,,,,,,,,


In [ ]:
import copy
import yaml

class MyDumper(yaml.Dumper):

    def increase_indent(self, flow=False, indentless=False):
        return super(MyDumper, self).increase_indent(flow, indentless)

test_template = {
    'description': 'Test similarity',
    'vars': {
        'chemname': 'Aspirin',
        'casrn': '50-78-2',
        'effect_type': 'Sub acute toxicity'
    },
    'assert': []
}

test_assert_template = {
        'type': 'assert-set',
        'threshold': 0.4,
        'assert':[{
                'type': 'similar',
                'value': 'Gastrointestinal bleeding',
                'threshold': 0.8
            },
            {
                'type': 'contains',
                'value': 'Gastrointestinal bleeding'
            }
        ]
    }

tests = []

for i, row in info.iterrows():
    for col in info.columns:
        if col in ['Other', 'Toxicity']: continue
        if row[col]:
            test_temp = copy.deepcopy(test_template)
            chemname, casrn = col.split(', ')
            test_temp['vars']['chemname'] = chemname
            test_temp['vars']['casrn'] = casrn
            if row['Toxicity']:
                test_temp['vars']['effect_type'] = f"{row['Toxicity']} toxicity"
            else:
                test_temp['vars']['effect_type'] = row['Other']
            for term in str(row[col]).split('; '):
                test_assert_temp = copy.deepcopy(test_assert_template)
                test_assert_temp['assert'][0]['value'] = term
                test_assert_temp['assert'][1]['value'] = term
                test_temp['assert'].append(test_assert_temp)
            tests.append(test_temp)

with open(Config.DIR_HOME / 'tests' / 'zero-shot-assertion' / 'tests.yaml', 'w') as outfile:
    yaml.dump(tests, outfile, Dumper=MyDumper, default_flow_style=False)


In [22]:
import copy
import yaml

class MyDumper(yaml.Dumper):

    def increase_indent(self, flow=False, indentless=False):
        return super(MyDumper, self).increase_indent(flow, indentless)

test_template = {
    'description': 'Test similarity',
    'vars': {
        'chemname': 'Aspirin',
        'casrn': '50-78-2',
        'effect_type': 'Sub acute toxicity'
    },
    'assert': [
        {
            'type': 'python', 
            'value': 'file://tests.py',
            'expected_phrases': ''
        }
    ]
}

tests = []

for i, row in info.iterrows():
    for col in info.columns:
        if col in ['Other', 'Toxicity']: continue
        if row[col]:
            test_temp = copy.deepcopy(test_template)
            chemname, casrn = col.split(', ')
            test_temp['vars']['chemname'] = chemname
            test_temp['vars']['casrn'] = casrn
            if row['Toxicity']:
                test_temp['vars']['effect_type'] = f"{row['Toxicity']} toxicity"
            else:
                test_temp['vars']['effect_type'] = row['Other']
            test_temp['assert'][0]['expected_phrases'] = str(row[col]).split('; ')
            tests.append(test_temp)

with open(Config.DIR_HOME / 'tests' / 'zero-shot-assertion' / 'tests.yaml', 'w') as outfile:
    yaml.dump(tests, outfile, Dumper=MyDumper, default_flow_style=False)
